**Project context:**

The city Saint Étienne, in France, is launching an initiative aimed at fostering the local network of research actors. This work aims to help the initiative, exploring the research activity in the city, discovering the degree of collaboration among local actors and exploring which fields or areas of research have been emerging in the last 6 years. The work has been carried through 6 tasks which involve:
1. data collection;
2. analysis of production indicators and trends in time;
3. analysis of the network of collaborations;
4. exploration of keywords;
5. detection of the diseases studied
6. clustering texts / topic modelling

In [ ]:
!pip install rake-nltk
!pip install yake
!pip install -U spacy
!pip install gensim
!pip install scispacy
!pip install https://s3-us-west-2.amazonaws.com/ai2-s2-scispacy/releases/v0.5.4/en_ner_bc5cdr_md-0.5.4.tar.gz

In [ ]:
import nltk
# Download a list of stopwords (common words) from nltk
nltk.download('stopwords')

# Download the tokenizer that splits sentences into words from nltk
nltk.download('punkt')
nltk.download('punkt_tab')


 **1.  Task 1: data extraction**

By using the OpenAlex open API, obtain all publications produced within Saint Étienne, from 2018 to 2023.

This task is completed following these steps:
1. Get all the **french** institutions.
2. From the retrieved results filter the institutions that are from **Saint Étienne**.
3. Next, get all the publications where the authors afiliations are within the institutions retrieved in the first step.

In [ ]:
#Code based on https://github.com/ourresearch/openalex-api-tutorials/blob/develop/notebooks/institutions/oa-percentage.ipynb and https://docs.openalex.org/api
import requests, json, pandas as pd

#dataframes display settings
pd.set_option('display.max_columns', None)
pd.set_option('display.width', None)

#first we query the Open Alex API
#input: endpoint name, filters for the query
#output: query request response results

def make_request(endpoint, filters):
  # put the URL together
  filtered_institutions_url = f'https://api.openalex.org/{endpoint}?filter={filters}'
  print(f'complete URL with filters:\n{filtered_institutions_url}')

  #get response
  response = requests.get(filtered_institutions_url, headers = {'User-agent': 'IRWA_1.0'}).json()
  results= response['results']

  # getting the total number of pages (results_count/responses_per_page)
  num_found=int(response['meta']['count'])
  resp_per_page=int(response['meta']['per_page'])
  num_pages = int(round(num_found/resp_per_page,0))
  print("number of pages: ",num_pages)
  print("number of items found: ",num_found)

  #get all the resposes going through each page
  for page in range(2, num_pages + 1):
    #print(page)
    results.extend(requests.get(filtered_institutions_url, params={'page': page}).json()['results'])

  #return all the items found
  return results

In [ ]:
# 1. make the request to the institutions endpoint to get only french institutions
# build the 'filter' parameter where we search french institutions
# Hint: use country_code based on the format explaned in https://github.com/ourresearch/openalex-api-tutorials/blob/develop/notebooks/institutions/oa-percentage.ipynb
filters = "YOUR CODE HERE"

# Make a request to the Open Alex API using the constructed filter and store the results in 'country_institutions'
country_institutions= make_request('institutions', filters)

# 2. get only the institutions from the city
# Initialize an empty dictionary to store institutions from the specified city
city_institutions={}

# Loop through each institution in the list of French institutions
for institution in country_institutions:
  # filter by city, look for the geonames_city_id in the geo field in the API documentation and introduce the code for Saint Étienne (2980291)
  # check the API documentation to understand better the institution object: https://docs.openalex.org/api-entities/institutions/institution-object
  if institution["YOUR CODE HERE"]["YOUR CODE HERE"]=='YOUR CODE HERE': #filter by city
    # If it matches, add the institution's ID as the key and its display name as the value to the 'city_institutions' dictionary
    city_institutions["YOUR CODE HERE"]=institution["YOUR CODE HERE"]

city_institutions_df=pd.DataFrame.from_dict(city_institutions, orient='index', columns=["Name"])
city_institutions_df.index.name='id'
city_institutions_df

In [ ]:
# 3. get publications where the author's institutions are from the city
#check how to define the filters needed here https://github.com/ourresearch/openalex-api-tutorials/blob/develop/notebooks/institutions/oa-percentage.ipynb and https://docs.openalex.org/api
# information needed: institutions id,  paratexts are not needed, starting and end dates of publication (last 6 years: from 2018 to 2023.)
city_works=[]
for institution in city_institutions:
  filters_works = ",".join((
   "YOUR CODE HERE"
  ))
  city_works.extend(make_request('works',filters_works))
print("total publications: ",len(city_works))

In [ ]:
# 4. We explore the publications retrieved
import pandas as pd
city_works_df = pd.DataFrame.from_dict(city_works)

In [ ]:
# store results in file
city_works_df.to_csv('city_works.csv',index=False)
# store results in file
city_institutions_df.to_csv('city_institutions.csv', index=True)
with open('city_works.json', 'w') as f:
    json.dump(city_works, f)
with open('city_institutions.json', 'w') as f:
    json.dump(city_institutions, f)

In [ ]:
import pandas as pd
import json
# Execute when data to download is not available
city_works_df =pd.read_csv('city_works.csv') #you must have the files provided uploaded in your local drive before this session
city_institutions_df =pd.read_csv('city_institutions.csv')
with open('city_works.json') as f:
    city_works = json.load(f)
with open('city_institutions.json') as f:
    city_institutions=json.load(f)
city_works_df

In [ ]:
city_institutions_df

In [ ]:
city_works_df.columns

**2. Task 2: time trends and production indicators**

For local decision makers, the evolution and specialisation of actors in certain scientific fields is a useful piece of information. Here we calculate the temporal evolution of the volume of publications.

Assumption:the granularity level is per year.

1.   first calculate the evolution of the volume of publications by counting the amount of existing publications per year and represeting them in a bar plot.
2. To complement the first part, represent the evolution of the volume of publications per year and publication category.
3.   Then, observe the evolution of the volume of publications per scientific domain. To do so explore the concepts in the publications metadata (level 0) and count the amount of publications corresponding to each scientifc domain per year.

In [ ]:
# 1.Here we consider the amount of publications done in total per year
df_summary_total= city_works_df.groupby(['publication_year']).size() #count publications per year
print(df_summary_total)
df_summary_total.plot(kind = 'bar') #plot

In [ ]:
# 2. In this section, we generate a summary of publications per year and type from the 'city_works_df' DataFrame
import matplotlib.pyplot as plt

# Group the 'city_works_df' DataFrame by both 'type' and 'publication_year', then count the occurrences.
# The result is reshaped using unstack() to have years as columns and publication types as rows.
df_summary_type= city_works_df.groupby(["YOUR CODE HERE","YOUR CODE HERE"]).size().unstack(level=1) #count publications per year and type
print(df_summary_type)

# Create a new figure for the plot
f = plt.figure()
# Plot the 'df_summary_type' DataFrame as a bar chart
df_summary_type.plot(kind = 'bar',ax=f.gca())
# Adjust the legend to be outside of the plot area on the right
plt.legend(loc='center left', bbox_to_anchor=(1.0, 0.5))
# Display the plot
plt.show()

In [ ]:
# 3. In this section, we analyze the number of publications per scientific domain and year in the 'city_works' list

# Initialize an empty list to store records of publications by their scientific domain and year
domain_records=[]

# Loop through each publication in the 'city_works' list
for publication in city_works:
  # Loop through each concept associated with the publication
  for concept in publication['concepts']:
    # Check if the concept's level is 0, which represents top-level scientific domains
    if concept["YOUR CODE HERE"]=="YOUR CODE HERE": #get scientific domains of level 0, take a look at the concept object https://docs.openalex.org/api-entities/concepts/concept-object
      domain_records.append([publication['publication_year'],concept['display_name'],1]) #count publications by category (the display name of the concept) and the publicationyear

# Convert the 'domain_records' list into a pandas DataFrame
pubs_cat_df = pd.DataFrame.from_dict(domain_records)
# Rename the DataFrame columns to 'Year', 'Scientific_domain', and 'publications'
pubs_cat_df.columns=['Year','Scientific_domain','publications']

# Group the 'pubs_cat_df' DataFrame by 'Scientific_domain' and 'Year', then count the occurrences.
# The result is reshaped using unstack() to have years as columns and scientific domains as rows.
df_summary_cat= pubs_cat_df.groupby(['Scientific_domain','Year']).size().unstack(level=1) #sum publications per year and type
print(df_summary_cat)

# Plot the 'df_summary_cat' DataFrame as a bar chart
f = plt.figure()
df_summary_cat.plot(kind = 'bar',ax=f.gca())
plt.legend(loc='center left', bbox_to_anchor=(1.0, 0.5))
plt.show()


**Task 3: Authors' collaboration network**

The collaboration network will help local actors understand how the city’s researchers are or have been establishing relationships. A collaboration occurs when two (or more) researchers affiliated to an organisation appear together in the same publication. Identify the ten most relevant/connected/central actors in the network resulting from the coauthorship of publications.

1.   For this part first get the authors with a Saint Étienne institution affiliation by checking the institutions of the authors (authorship) and keeping only those belonging to our prior list of institutions ofthe city.
2.   Then make pairs between the coauthors in each publication that belong to a city institution.
3. Using networkx build an undirected graph of author's collaborations, calculate several measures like the degree centrality, closeness centrality, betweeness centrality, eigen vector centrality (suitable for undirected graphs) and page rank.
4. Get the top 10 authors with the best scores according to each centrality measure calculated.



    Degree Centrality:
        What It Measures: The number of direct connections (collaborations) an author has.
        Interpretation: An author with high degree centrality has collaborated with many different authors. This might indicate that the author is versatile, active in the community, or works on interdisciplinary projects.

    Closeness Centrality:
        What It Measures: The average "distance" (shortest path) from one author to all other authors in the network.
        Interpretation: An author with high closeness centrality can reach any other author in the network through a small number of collaborations. This might suggest that the author is centrally located in the academic community and has a broad reach.

    Betweenness Centrality:
        What It Measures: The number of times an author acts as a "bridge" along the shortest path between two other authors.
        Interpretation: An author with high betweenness centrality connects disparate parts of the network. They may be influential in bringing together researchers from different subfields or might work on niche areas that connect broader topics.

    Eigenvector (Eigen) Centrality:
        What It Measures: The influence of an author in the network, taking into account the quality (or centrality) of their collaborators.
        Interpretation: An author with high eigen centrality not only collaborates often but collaborates with influential authors. It's like being a member of an "elite club" of top researchers.

    PageRank:
        What It Measures: Originally designed for web pages, in the context of coauthors, it measures the likelihood that someone randomly navigating collaborations ends up with a particular author. It considers both direct collaborators and the importance (or rank) of those collaborators.
        Interpretation: Similar to Eigen centrality, but with a different algorithmic approach, a high PageRank indicates that an author collaborates with influential authors. Being frequently co-authored with or referenced by other influential authors boosts one's PageRank.

In summary, while all these measures give insight into an author's prominence or influence in a coauthorship network, each offers a distinct perspective. Degree centrality focuses on the volume of collaborations, closeness on the reach within the academic community, betweenness on the ability to connect different researchers, and eigenvector centrality and PageRank on the quality or influence of collaborators.

In [ ]:
# 1. In this section, we extract the authors who have affiliations with institutions from the city 'Sainte'

import itertools
import csv

# Initialize a list to store pairs of authors who have co-authored a publication
relations_pairs = []
# Create a dictionary to map author IDs to their display names, for later reference
authors_ids_names_map = {} #for later use to display names of the top authors

# Loop through each publication in the 'city_works' list
for publication in city_works:
  authors = [] # A temporary list to store authors of the current publication
  # Loop through each authorship in the publication
  for author in "YOUR CODE HERE": #get the authorships in the publication
    # Loop through each institution affiliated with the author
    for institution in "YOUR CODE HERE":#get the institution of the author
      # Check if the institution's ID matches any of the city institutions
      if "YOUR CODE HERE" in list("YOUR CODE HERE"): #verify if author corresponds to a city institution
        # Append the author's ID to the 'authors' list
        authors.append(author['author']['id']) #get ids to avoid missing the same authors with different name displays
        # If the author's ID is not already in the mapping dictionary, add it
        if author['author']['id'] not in authors_ids_names_map:
          authors_ids_names_map[author['author']['id']]=author['author']['display_name']
  # Generate all possible combinations (pairs) of authors from the 'authors' list
  for pair in itertools.combinations(authors,2): #create pairs of authors to define edges
    relations_pairs.append(pair)


# 2. In this section, we replace the author IDs with their display names in the pairs

names_pairs = []  # Initialize a list to store pairs with author names

# Loop through each pair in 'relations_pairs'
for pair in relations_pairs:
    # Replace the IDs in the pair with their corresponding display names
    new_pair=(authors_ids_names_map[pair[0]],authors_ids_names_map[pair[1]])
    names_pairs.append(new_pair)
# Remove duplicate pairs by considering (a,b) and (b,a) as the same
pairs=set(tuple(sorted(p)) for p in names_pairs)
print("edges:",len(pairs))

In [ ]:
#Export pairs to plot the graph with any tool you'd like
with open('graph_input.csv','w') as out:
    csv_out=csv.writer(out)
    csv_out.writerow(['author_1','author_2'])
    for row in pairs:
        csv_out.writerow(row)

In [ ]:
import networkx as nx
import operator
import matplotlib.pyplot as plt

#see more about networkx: https://networkx.org/documentation/stable/tutorial.html
G = nx.Graph()  # create empty graph
G.add_edges_from(pairs) # create graph from edges

#degree
#This tells the top 10 authors with most co-authors from institutions in Sainte

print("Top 10 based in the degree:\n")
counter=0
#find the graph degree using the degree function
for author in sorted("YOUR CODE HERE", key=lambda x: x[1], reverse=True)[:10]:
  counter+=1
  print(counter,author[0],author[1])

In [ ]:
def print_top_10(centrality_scores):
  cent_vals= dict(sorted(centrality_scores.items(), key=operator.itemgetter(1), reverse=True)[:10])
  counter=0
  for author in cent_vals:
    counter+=1
    print(counter,author,cent_vals[author])

#degree centrality (same as considering the degree)
print("degree centrality")
deg_centrality = "YOUR CODE HERE" #Get the degree centrality of the network
print_top_10(deg_centrality)


In [ ]:
#closeness centrality
#this is a measure of closeness or distance to others in the graph, we see authors that are the closest to other authors.
#Notice that higher values of closeness indicate higher centrality.
print("closeness centrality")
close_centrality = "YOUR CODE HERE" #get the closeness of the network
print_top_10(close_centrality)

In [ ]:
#Betweeness centrality (might take some time (~3min) to calculate, no worries....)
#this measures how often a node is in the shortest path between two other nodes in the network
bet_centrality = "YOUR CODE HERE" #get the betweenness_centrality
print_top_10(bet_centrality)

In [ ]:
#we calculate the eigenvector centrality which is suitable for undireced graphs
#A high eigenvector score means that a node is connected to many nodes who themselves have high scores
eigen_centrality = "YOUR CODE HERE"
print_top_10(eigen_centrality)

In [ ]:
#page rank
#this measures the importance of a node in a graph (popularity), this measure is more suitable for directed graphs so eigen centrality is better for our case
pr = "YOUR CODE HERE"
print_top_10(pr)

**Task 4: Exploring keywords**

It is
often necessary to get to a deeper level of understanding of the content of scientific publications. Thus,
for this, we extract keywords from the abstracts of the publications.
1.  extract keywords from all the available abstracts and titles, to do so, first rebuild the abstracts from the inverted indexes provided and concatenate (title + abstract). Then, use 2 approaches to get the keywords of each publication: Rake-nltk and Yake.

2.   identify the most recurrent keywords attached to the publications of local actors. For this step, using the top 10 keywords obtained with yake per publication, integrate all the relevant keywords and calculate their frequency. Then order these keywords based on their frequency to identify the most common keywords considering all the publications.

3. Use also an approach based on the frequency (tf-idf model) of the 1-3 grams used in all the abstracts.




In [ ]:
#preprocessing
def clean_text(text):
    import numpy as np
    import re
    if type(text) == float:
        return ""
    temp = text.lower()
    temp = temp.replace("'","") # to avoid removing contractions in english
    temp = re.sub("@[A-Za-z0-9_]+","", temp)
    temp= re.sub(r'[^\w]', ' ', temp)
    temp = re.sub('[()!?]', ' ', temp)
    temp = re.sub(r'\[.*?\]',' ', temp)
    temp = temp.replace("\n"," ")
    temp = temp.replace("."," ")
    temp = temp.replace("mml","")
    temp = temp.replace("mrow","")
    temp = temp.replace("msub","")
    temp = temp.replace("http","")
    temp = temp.replace("www","")
    temp = temp.replace("mathml","")
    temp = temp.replace("xlink","")
    temp = temp.replace("\""," ")
    temp = re.sub('[0-9]', '', temp)
    temp = temp.strip()
    temp=re.sub(' +', ' ', temp)
    return temp

In [ ]:
#first we build back the abstracts (including the title)
def build_abtracts(publications):
  docs={}
  for publication in publications:
    publication_terms={}
    if publication['abstract_inverted_index'] and publication['title']:
      for term in publication['abstract_inverted_index']:
        for value in publication['abstract_inverted_index'][term]:
          publication_terms[int(value)]=term
      ordered_values= "YOUR CODE HERE" #rebuild the text by sorting its terms by key (hint: use 'sorted')
      #print(publication['title'])
      text= publication['title']
      for element in ordered_values:
        text+=publication_terms[element]+" "
      docs[publication['id']]=clean_text(text)
  return docs

In [ ]:
docs=build_abtracts(city_works)

In [ ]:
#first apporach to get relevant keywords per publication
#get keywords using rake: RAKE short for Rapid Automatic Keyword Extraction algorithm, is a domain independent keyword extraction algorithm which tries to determine key phrases in a body of text by analyzing the frequency of word appearance and its co-occurance with other words in the text.
#this approach can get as results long sentences
#Rake is a python implementation of the algorithm as mentioned in paper Automatic keyword extraction from individual documents by Stuart Rose, Dave Engel, Nick Cramer and Wendy Cowley
#After every candidate keyword is identified and the graph of word co-occurrences is complete, a score is calculated for each candidate keyword.
#and defined as the sum of its member word scores.
import nltk

# Import the Rake algorithm from the rake_nltk library
from rake_nltk import Rake

# Initialize the Rake algorithm
rake_nltk_var = Rake()

# Create an empty dictionary to store the top keywords for each document
keywords_doc = {}

# Loop through each document in the docs dictionary
for doc in docs:
    # Use the Rake algorithm to extract keywords from the current document
    rake_nltk_var.extract_keywords_from_text(docs[doc])

    # Get the top 10 ranked phrases (keywords) from the extracted results
    keyword_extracted = rake_nltk_var.get_ranked_phrases()[:10]

    # Store the top keywords in the keywords_doc dictionary with the document's name as the key
    keywords_doc[doc] = keyword_extracted

# Display the dictionary containing top keywords for each document
# Get a sample of keys:
sample_keys = list(keywords_doc.keys())[:5]
print(sample_keys)

for key in sample_keys:
  print(keywords_doc[key])


RAKE (Rapid Automatic Keyword Extraction) Algorithm Explanation:

    Preprocessing:
        RAKE starts by splitting the text into individual sentences. This is often done using punctuation and newline characters as delimiters.
        Each sentence is then tokenized, i.e., broken down into individual words.

    Stopword Removal:
        Stopwords (common words like "and", "the", "is", etc.) are removed from these words since they're frequent across all documents and don't provide specific meaning in keyword extraction.

    Phrase Partitioning:
        After removing the stopwords, the words that are left behind are used to split the text into phrases. The idea is that stopwords and punctuation often delimit meaningful phrases. So, "performance of neural networks" might become the phrase "performance neural networks".

    Calculating Word Scores:
        Each word's score is computed based on its frequency (how often it appears) and its degree (how many other words it's associated with in the phrases).
        Typically, the score is calculated as: score(word)=degree(word)/frequency(word)score(word)=degree(word)/frequency(word).
        Words that appear frequently alongside many other words have a high score.

    Assigning Scores to Phrases:
        The score for a phrase is just the sum of scores of the words it contains.
        Using the earlier example, the score for "performance neural networks" would be the sum of the scores for "performance", "neural", and "networks".

    Ranking and Extraction:
        Finally, the phrases are ranked by their scores. The top-scoring phrases are the key phrases or keywords extracted by the algorithm.

Why RAKE works:
RAKE is based on the observation that keywords and key phrases often have words that appear together frequently and are not frequently used across the entire document. By contrast, stopwords (like "and", "to", etc.) appear very frequently and in conjunction with many other words. By removing these stopwords and looking at the frequency and co-occurrence patterns of the remaining words, RAKE identifies keywords and phrases that are deemed important to the document.

In essence, RAKE is a simple yet effective unsupervised method for keyword extraction from individual documents without needing any training data. It's computationally efficient and works well for many keyword extraction tasks.

In [ ]:
#Second apporach using yake based on Campos, R., Mangaravite, V., Pasquali, A., Jatowt, A., Jorge, A., Nunes, C. and Jatowt, A. (2020). YAKE! Keyword Extraction from Single Documents using Multiple Local Features. In Information Sciences Journal. Elsevier, Vol 509, pp 257-289. pdf
#YAKE! is a light-weight unsupervised automatic keyword extraction method which rests on text statistical features extracted from single documents to select the most important keywords of a text.
#I assume all abstracts are in english beside having observed publications in french

# Import the yake library for keyword extraction.
import yake

# Create an instance of the KeywordExtractor class from the yake library.
kw_extractor = yake.KeywordExtractor()

# Initialize an empty dictionary to store the extracted keywords for each document.
all_docs_keywords = {}

# Loop through each document in the 'docs' dictionary.
for doc in docs:

    # For each document, create a custom keyword extractor with specific parameters:
    # - Language set to English ("en").
    # - Extract keywords ranging from 1 word up to 3 words (n-grams, where n=3).
    # - Deduplication threshold set to 0.9 (dedupLim=0.9); keywords with similarity above this will be considered duplicates.
    # - Return the top 10 keywords (top=10).
    # - No additional features specified (features=None).
    custom_kw_extractor = yake.KeywordExtractor(lan="en", n=3, dedupLim=0.9, top=10, features=None)

    # Use the custom keyword extractor to extract keywords from the current document.
    keywords = custom_kw_extractor.extract_keywords(docs[doc])

    # Store the extracted keywords in the 'all_docs_keywords' dictionary using the document's name as the key.
    all_docs_keywords[doc] = keywords

# Display the dictionary containing extracted keywords for each document.
sample_keys = list(all_docs_keywords.keys())[:5]
print(sample_keys)

for key in sample_keys:
  print(all_docs_keywords[key])

YAKE (Yet Another Keyword Extractor) is a keyword extraction algorithm that, like RAKE, is unsupervised and does not require training. However, YAKE uses a different methodology and introduces several improvements over algorithms like RAKE. Let's delve into how YAKE works:

YAKE (Yet Another Keyword Extractor) Algorithm Explanation:

    Preprocessing:
        YAKE starts by converting the entire text to lowercase.
        The text is then tokenized into individual words.

    Candidate Keyword Generation:
        YAKE generates a set of candidate keywords. These are composed of unigrams (single words), bigrams (two consecutive words), up to n-grams (n consecutive words), where nn is usually a small number like 3 or 4.

    Computing Features for Each Candidate:
        For each candidate keyword, YAKE computes several features:
            Term Frequency (TF): The number of times the candidate keyword appears in the document.
            Document Frequency (DF): Reflects how common or rare a word is across several documents. For a single document, this is often taken from a pre-computed corpus or set to 1.
            Candidate Casing Feature: Helps in identifying proper nouns or terms that frequently start with a capital letter.
            Positional Feature: Gives a sense of where in the document the candidate keyword appears. Words appearing at the beginning or end might be treated differently than those in the middle.

    Computing a Score for Each Candidate:
        YAKE uses the features computed to calculate a score for each candidate keyword. The idea is to give lower scores to more important keywords.
        One of the main differences between YAKE and other keyword extractors is how it calculates this score. It considers both how often the term appears (TF) and how "unique" the term is (using a variation of the DF). It also takes into account the length of the candidate keyword (longer terms are usually penalized) and its positional information.

    Ranking and Extraction:
        Finally, YAKE ranks the candidate keywords based on their scores and returns the top-ranked ones as the extracted keywords.

Why YAKE works:
The intuition behind YAKE is that important keywords in a document are both relevant (appear frequently in the document) and distinctive (don't appear too commonly in all documents). By considering multiple features and taking into account the uniqueness of terms, YAKE can effectively identify and rank keywords in a given document.

In essence, YAKE provides a balanced approach to keyword extraction by not just relying on term frequency but also on the distinctiveness and contextual information of terms. This makes it effective for extracting keywords from individual documents, even in the presence of noise or when the document is short.

TF-IDF (Term Frequency-Inverse Document Frequency) and YAKE (Yet Another Keyword Extractor) are both methods used to extract important terms or keywords from documents, but they approach the task differently. Here's a comparison between the two methods:

    Approach:
        TF-IDF: This method is based on the frequency of a term in a document (TF) balanced against the number of documents that contain the term (IDF). If a term appears frequently in a particular document but is rare across other documents, it will have a high TF-IDF score.
        YAKE: YAKE takes into consideration not just the term's frequency but also other features like the term's position in the document, the casing of the term, and its degree of uniqueness across a larger corpus.

In [ ]:
#get top keywords considering all publications based on the top keywords of the prior approach (Yake)
top_keywords_overall={}
for publication in all_docs_keywords:
  for pair in all_docs_keywords[publication]:
    if pair[0] not in top_keywords_overall:
      top_keywords_overall[pair[0]]=0
    top_keywords_overall[pair[0]]+=1
top_keywords=dict(sorted(top_keywords_overall.items(), key=lambda item: item[1],reverse=True))
top_keywords

In [ ]:
#approach based on tfidf vectorization
from nltk.translate.ribes_score import ngrams
from sklearn.feature_extraction.text import TfidfVectorizer
import numpy as np

def get_tfidf_top_terms(docs,ngram_range,n_top):
  tfidf_vectorizer = TfidfVectorizer(min_df=0.01, ngram_range=ngram_range,  stop_words='english') #don't consider unigrams as not so relevant terms are most likely to appear
  tfidf = tfidf_vectorizer.fit_transform(docs)
  importance = np.argsort(np.asarray(tfidf.sum(axis=0)).ravel())[::-1]
  tfidf_feature_names = np.array(tfidf_vectorizer.get_feature_names_out())
  return tfidf_feature_names[importance[:n_top]]


In [ ]:
#get top terms considering all publications using the tf-idf approach for 2-3 grams and for 1-3 grams
print("2-3 grams")
for element in get_tfidf_top_terms(docs.values(),(2,3),20):
  print(element)
print("\n\n1-3 grams")
for element in get_tfidf_top_terms(docs.values(),(1,3),20):
  print(element)


**Task 5: identifying the most studied diseases in the city**

Policymakers want to understand which diseases are the most studied by local researchers. For this task, we identify the most studied diseases in scientific
publications by identifying diseases present in the textual content of the abstracts.
We identify disease names from the abstract of publications.
1.   filter the publications in OpenAlex by the concept Medicine (id = c71924100). This is done through filtering the Sainte publications by the given concept.
2.   identify disease names. To do this get the abstracs (title + abstract) of the medicine publications. Then use the spacy library as a named entity recognition tool. Use the n_ner_bc5cdr_md model which was trained to detect disease names. This model is applied to each publication's abstract. Once you we have extracted the diseases from all the publications, count the frequecy of each disease to identify the illness most addressed.

In [ ]:
#We first filter the publications by the concept Medicine (id = c71924100)
medicine_publications=[]
for publication in city_works:
  for concept in publication['concepts']:
    if "YOUR CODE HERE" in "YOUR CODE HERE": #get medicine publications where concept id = c71924100
      medicine_publications.append(publication)

In [ ]:
#get diseases per publication
import scispacy
import spacy
import en_ner_bc5cdr_md
from spacy import displacy
from scispacy.abbreviation import AbbreviationDetector
from scispacy.umls_linking import UmlsEntityLinker

##now we get the abstracts (title + abstract) for the medicine publications
medicine_pubs="YOUR CODE HERE" #build the abstracts for medicine_publications

#Load the model
nlp = en_ner_bc5cdr_md.load()
entities_docs={}

for doc in medicine_pubs:
  doc_diseases=[]
  entities = nlp(medicine_pubs[doc])
  for ent in entities.ents:
    #print(ent.label_)
    if ent.label_ in ["DISEASE"]: #if the entity is a disease
        doc_diseases.append(ent.text)
  single_diseases = list(dict.fromkeys(doc_diseases)) #remove duplicates

  entities_docs[doc]=single_diseases
entities_docs

In [ ]:
# Import the 'itertools' library, which provides various functions that work on iterators to produce complex iterators.
# Note: In this code snippet, 'itertools' is imported but not actually used.
import itertools

# Initialize an empty dictionary to count the occurrences of each disease across all publications.
diseases_count = {}

# Loop through each publication in the 'entities_docs' dictionary.
for publication in entities_docs:

    # Loop through each disease entity identified in the current publication.
    for disease in entities_docs[publication]:
        # If the disease hasn't been encountered before, initialize its count to 0.
        if disease not in diseases_count:
            diseases_count[disease] = 0

        # Increment the count for the current disease.
        diseases_count[disease] += 1

# Sort the 'diseases_count' dictionary by the count of each disease in descending order.
diseases_ordered = dict(sorted(diseases_count.items(), key=lambda item: item[1], reverse=True))

# Display the sorted dictionary.
print(diseases_ordered)

# Extract the top 50 diseases from the ordered dictionary.
top_dis = {A: N for (A, N) in [x for x in diseases_ordered.items()][:50]}

# Convert the 'top_dis' dictionary into a pandas DataFrame.
# The disease names will be the index and the counts will be the values.
df_diseases = pd.DataFrame.from_dict(top_dis, orient='index')

# The resulting DataFrame is displayed as the output.
df_diseases

**task 5: clustering texts / topic modelling**


Beyond the extraction of keywords, it would be useful to cluster the texts of the abstracts in accordance with their content. This exercise enables one in practice to extract topics emerging from the corpus of identified documents.
1.   show what are the textual terms that characterise each cluster. To do so use LDA as a topic modeling apporach applied over all the Sainte publications. Define 10 topics to be created and display the top terms that represent each topic.


LDA Explained:

Imagine you're given a collection of research papers but without any titles or topics mentioned. Your task is to categorize these papers into different topics or fields. LDA is a statistical tool that helps you do this.

    Initialization:
        Decide on the number of topics you believe exist in the collection. This is similar to saying, "I think these papers can be grouped into, say, 5 main fields."
        Randomly assign each word in each paper to one of these topics. This initial assignment is likely incorrect but serves as a starting point.

    Iterative Refinement:
        For each paper, and for each word in that paper, LDA reviews the word's assigned topic, considering two main criteria:
            How often does the topic occur in that paper? (Is "Neural Networks" a dominant topic in this paper?)
            How often does the word appear across all papers for that topic? (How often is the word "neuron" associated with the "Neural Networks" topic across all papers?)
        Based on this information, the model may decide to reassign the word to a different topic.

    Convergence:
        This process of reassigning words and refining topics is repeated many times. As it progresses, the assignments become more accurate.
        Eventually, the algorithm converges, meaning that the topics and their associated words don't change much between iterations.

    Output:
        Once LDA is done, each paper will have a distribution of topics (e.g., Paper 1 might be 70% about Neural Networks, 20% about Genetic Algorithms, and 10% about Quantum Computing).
        Additionally, each topic will have a list of words most associated with it, helping you understand the theme or content of that topic.

In essence, LDA is like a smart librarian. Given a stack of unlabeled books, the librarian tries to categorize them into different genres by looking at which terms frequently appear together in each book. Through iterative sorting and refining, the librarian gets better and better at this categorization until she's confident about the main genres and the books that belong to each.

In [ ]:
# Import the 'gensim' library, which is used for topic modeling and document similarity analysis.
import gensim

# Import 'simple_preprocess' from 'gensim'. This utility function helps in tokenizing and cleaning text.
from gensim.utils import simple_preprocess

# Import the Natural Language Toolkit (nltk) library.
import nltk

# Import 'LatentDirichletAllocation' from 'sklearn'. This is the main LDA model.
from sklearn.decomposition import LatentDirichletAllocation

# Import 'CountVectorizer' from 'sklearn'. This is used to transform the text data into a bag-of-words representation.
from sklearn.feature_extraction.text import CountVectorizer

# Call a function 'build_abstracts' (which seems to be defined elsewhere) to get the abstracts of the 'city_works'.
city_abstracts = build_abtracts(city_works)

# Set the number of clusters/topics.
n_components = 10

# Set the number of top words to be extracted for each topic.
n_top_words = 20

# Initialize the CountVectorizer with the following parameters:
# - max_df: Words that appear in more than 95% of the documents are ignored.
# - min_df: Words that appear in less than 1% of the documents are ignored.
# - stop_words: Common English words ('and', 'the', etc.) are removed.
# - ngram_range: Consider unigrams (single words) and bigrams (two consecutive words).
count_vectorizer = CountVectorizer(max_df=0.95, min_df=0.01, stop_words='english', ngram_range=(1, 2))

# Transform the city_abstracts into a bag-of-words representation using the CountVectorizer.
tf = count_vectorizer.fit_transform(city_abstracts.values())

# Initialize and fit the LDA model to the bag-of-words representation:
# - n_components: Number of topics/clusters to be generated.
# - random_state: Seed for reproducibility.
lda = LatentDirichletAllocation(n_components=n_components, random_state=1).fit(tf)

In [ ]:
def get_model_topics(model, vectorizer, n_top_words=n_top_words):
    word_dict = {}
    feature_names = vectorizer.get_feature_names_out()
    for topic_idx, topic in enumerate(model.components_):
        top_features_ind = topic.argsort()[:-n_top_words - 1:-1]
        top_features = [feature_names[i] for i in top_features_ind]
        word_dict[topic_idx] = top_features

    return pd.DataFrame(word_dict)

In [ ]:
get_model_topics(lda, count_vectorizer, n_top_words=n_top_words)